# LightGBM

In [ ]:
%pip install -q mlflow lightgbm scikit-learn pandas matplotlib

In [ ]:
import os, sys
if "google.colab" in sys.modules:
    pass
sys.path.insert(0, os.getcwd())

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

from src.data import load_raw, submission_ids
from src.features import WalmartFeatureBuilder, MARKDOWN_COLS
from src.metrics import wmae, holiday_weights
from src.validation import year_back_split, time_folds, DEV_TRAIN_END, TRAIN_END
from src.tracking import setup_mlflow, REGISTERED_MODEL_NAME

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("MLflow tracking:", setup_mlflow("LightGBM_Training"))

In [ ]:
raw = load_raw()
train, test = raw["train"], raw["test"]

tr_idx, va_idx = year_back_split(train["Date"])
dev_train, dev_valid = train.iloc[tr_idx], train.iloc[va_idx]
print(f"train: {train.shape}, test: {test.shape}")
print(f"dev_train: {len(dev_train):,} რიგი (<= {DEV_TRAIN_END.date()}), "
      f"dev_valid: {len(dev_valid):,} რიგი (39 კვირა, შეიცავს Thanksgiving/Christmas/Super Bowl-ს)")

In [ ]:
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    mlflow.log_params({
        "negative_sales_policy": "keep", "markdown_nan_policy": "fill 0 + flag",
        "cpi_unemployment_policy": "per-store ffill", "feature_builder": "src.features.WalmartFeatureBuilder",
    })

In [ ]:
def make_pipe(**lgbm_params):
    params = dict(n_estimators=2000, learning_rate=0.05, num_leaves=127,
                  objective="l1", subsample=0.9, colsample_bytree=0.8,
                  min_child_samples=20, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    params.update(lgbm_params)
    return Pipeline([
        ("features", WalmartFeatureBuilder(features_df=raw["features"],
                                           stores_df=raw["stores"])),
        ("model", LGBMRegressor(**params)),
    ])

def eval_year_back(pipe):
    pipe.fit(dev_train, dev_train["Weekly_Sales"],
             model__sample_weight=holiday_weights(dev_train["IsHoliday"]))
    pred = pipe.predict(dev_valid.drop(columns=["Weekly_Sales"]))
    return wmae(dev_valid["Weekly_Sales"], pred, dev_valid["IsHoliday"])

with mlflow.start_run(run_name="LightGBM_Baseline"):
    # baseline
    base_wmae = eval_year_back(make_pipe())
    mlflow.log_metric("valid_wmae", base_wmae)
print(f"Baseline (l1, 127 leaves) valid WMAE: {base_wmae:.0f}")

In [ ]:
param_grid = [
    dict(objective="l1", num_leaves=63),
    dict(objective="l1", num_leaves=255, min_child_samples=40),
    dict(objective="l2", num_leaves=127),
    dict(objective="huber", num_leaves=127),
]

best_cv, best_params = None, None
with mlflow.start_run(run_name="LightGBM_CV"):
    # objective-ebis shedareba
    for i, p in enumerate(param_grid):
        with mlflow.start_run(run_name=f"config_{i}", nested=True):
            fold_scores = []
            for tri, vai in time_folds(train["Date"], n_folds=3, valid_weeks=13):
                ftr, fva = train.iloc[tri], train.iloc[vai]
                pipe = make_pipe(**p)
                pipe.fit(ftr, ftr["Weekly_Sales"],
                         model__sample_weight=holiday_weights(ftr["IsHoliday"]))
                pred = pipe.predict(fva.drop(columns=["Weekly_Sales"]))
                fold_scores.append(wmae(fva["Weekly_Sales"], pred, fva["IsHoliday"]))
            cv_mean = float(np.mean(fold_scores))
            mlflow.log_params(p)
            mlflow.log_metric("cv_wmae_mean", cv_mean)
            print(f"config_{i}: {p} -> CV WMAE {cv_mean:.0f}")
            if best_cv is None or cv_mean < best_cv:
                best_cv, best_params = cv_mean, p

print(f"\nსაუკეთესო: {best_params} (CV WMAE {best_cv:.0f})")

In [ ]:
with mlflow.start_run(run_name="LightGBM_Final") as run:
    final_pipe = make_pipe(**best_params)
    final_wmae = eval_year_back(final_pipe)
    mlflow.log_params(best_params)
    mlflow.log_metric("valid_wmae", final_wmae)

    names = final_pipe.named_steps["features"].get_feature_names_out()
    imp = pd.Series(final_pipe.named_steps["model"].feature_importances_,
                    index=names).sort_values()
    fig, ax = plt.subplots(figsize=(7, 9))
    imp.plot.barh(ax=ax)
    ax.set_title("LightGBM feature importance (splits)")
    plt.tight_layout()
    mlflow.log_figure(fig, "feature_importance.png")
    plt.show()

    final_pipe.fit(train, train["Weekly_Sales"],
                   model__sample_weight=holiday_weights(train["IsHoliday"]))
    mlflow.sklearn.log_model(final_pipe, name="model", code_paths=["src"],
                             serialization_format="cloudpickle")
    lgbm_final_run_id = run.info.run_id

print(f"საბოლოო valid WMAE: {final_wmae:.0f} | run: {lgbm_final_run_id}")

In [ ]:
REGISTER_AS_CHAMPION = False

if REGISTER_AS_CHAMPION:
    from mlflow import MlflowClient
    mv = mlflow.register_model(f"runs:/{lgbm_final_run_id}/model", REGISTERED_MODEL_NAME)
    MlflowClient().set_registered_model_alias(REGISTERED_MODEL_NAME, "champion", mv.version)
    print(f"დარეგისტრირდა: {REGISTERED_MODEL_NAME} v{mv.version} (alias: champion)")